# MTG Source Stack Walkthrough

This notebook walks through the local-first workflow built in this repo so far:

1. create a clean SQLite database
2. import sample Scryfall and MTGJSON data
3. inspect the imported catalog tables
4. create a personal inventory
5. add owned cards
6. value the collection from imported price snapshots

It uses tiny local sample payloads so the whole walkthrough is easy to rerun.

## Setup

Run the notebook from top to bottom so the scratch database is recreated in a clean state.

This cell finds the repo root, forces the notebook to use the repo copy of
`mtg_source_stack` instead of any stale installed copy, and creates a scratch
folder for the walkthrough outputs.


In [ ]:
from __future__ import annotations

import importlib
import json
import shutil
import sqlite3
import subprocess
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "MtG Source Stack").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
STACK_DIR = REPO_ROOT / "MtG Source Stack"

STACK_DIR_STR = str(STACK_DIR)
sys.path = [path for path in sys.path if path != STACK_DIR_STR]
sys.path.insert(0, STACK_DIR_STR)

for module_name in list(sys.modules):
    if module_name == "mtg_source_stack" or module_name.startswith("mtg_source_stack."):
        del sys.modules[module_name]

mvp_importer = importlib.import_module("mtg_source_stack.mvp_importer")
personal_inventory_cli = importlib.import_module("mtg_source_stack.personal_inventory_cli")

initialize_database = mvp_importer.initialize_database
import_scryfall_cards = mvp_importer.import_scryfall_cards
import_mtgjson_identifiers = mvp_importer.import_mtgjson_identifiers
import_mtgjson_prices = mvp_importer.import_mtgjson_prices

create_inventory = personal_inventory_cli.create_inventory
list_inventories = personal_inventory_cli.list_inventories
search_cards = personal_inventory_cli.search_cards
add_card = personal_inventory_cli.add_card
list_owned = personal_inventory_cli.list_owned
valuation = personal_inventory_cli.valuation
render_table = personal_inventory_cli.render_table
format_add_card_result = personal_inventory_cli.format_add_card_result
format_owned_rows = personal_inventory_cli.format_owned_rows

WORK_DIR = STACK_DIR / "_walkthrough_output"
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

DB_PATH = WORK_DIR / "walkthrough.db"
SCRYFALL_JSON = WORK_DIR / "scryfall_sample.json"
IDENTIFIERS_JSON = WORK_DIR / "identifiers_sample.json"
PRICES_JSON = WORK_DIR / "prices_sample.json"

print("repo root:", REPO_ROOT)
print("stack dir:", STACK_DIR)
print("work dir:", WORK_DIR)
print("importer module:", mvp_importer.__file__)
print("inventory cli module:", personal_inventory_cli.__file__)


## Build Tiny Sample Data Files

The real importer expects local Scryfall and MTGJSON bulk files. For the
walkthrough, we write tiny sample JSON payloads that mimic the important parts
of those bulk files.

In [ ]:
scryfall_payload = [
    {
        "id": "s1",
        "oracle_id": "o1",
        "name": "Lightning Bolt",
        "set": "lea",
        "set_name": "Limited Edition Alpha",
        "collector_number": "161",
        "lang": "en",
        "rarity": "common",
        "released_at": "1993-08-05",
        "mana_cost": "{R}",
        "type_line": "Instant",
        "oracle_text": "Lightning Bolt deals 3 damage to any target.",
        "colors": ["R"],
        "color_identity": ["R"],
        "finishes": ["nonfoil"],
        "legalities": {"commander": "legal"},
        "purchase_uris": {"tcgplayer": "https://example.test/tcg/lightning-bolt"},
        "tcgplayer_id": 534658,
        "cardmarket_id": 752712,
    },
    {
        "id": "s2",
        "oracle_id": "o2",
        "name": "Sol Ring",
        "set": "clb",
        "set_name": "Commander Legends: Battle for Baldur's Gate",
        "collector_number": "860",
        "lang": "en",
        "rarity": "rare",
        "released_at": "2022-06-10",
        "mana_cost": "{1}",
        "type_line": "Artifact",
        "oracle_text": "{T}: Add {C}{C}.",
        "colors": [],
        "color_identity": [],
        "finishes": ["nonfoil", "foil"],
        "legalities": {"commander": "legal"},
        "purchase_uris": {"tcgplayer": "https://example.test/tcg/sol-ring"},
        "tcgplayer_id": 271111,
        "cardmarket_id": 665555,
    },
]

identifiers_payload = {
    "data": {
        "uuid-1": {
            "scryfallId": "s1",
            "tcgplayerProductId": "534658",
            "cardKingdomId": "12345",
            "mcmId": "752712",
            "cardsphereId": "98765",
        },
        "uuid-2": {
            "scryfallId": "s2",
            "tcgplayerProductId": "271111",
            "cardKingdomId": "54321",
            "mcmId": "665555",
            "cardsphereId": "56789",
        },
    }
}

prices_payload = {
    "data": {
        "uuid-1": {
            "paper": {
                "tcgplayer": {
                    "currency": "USD",
                    "retail": {"normal": {"2026-03-27": 2.92}},
                    "buylist": {"normal": {"2026-03-27": 1.10}},
                },
                "cardmarket": {
                    "currency": "EUR",
                    "retail": {"normal": {"2026-03-27": 1.51}},
                },
            }
        },
        "uuid-2": {
            "paper": {
                "tcgplayer": {
                    "currency": "USD",
                    "retail": {
                        "normal": {"2026-03-27": 1.75},
                        "foil": {"2026-03-27": 4.25},
                    },
                    "buylist": {
                        "normal": {"2026-03-27": 0.75},
                        "foil": {"2026-03-27": 2.00},
                    },
                }
            }
        },
    }
}

SCRYFALL_JSON.write_text(json.dumps(scryfall_payload), encoding="utf-8")
IDENTIFIERS_JSON.write_text(json.dumps(identifiers_payload), encoding="utf-8")
PRICES_JSON.write_text(json.dumps(prices_payload), encoding="utf-8")

print(SCRYFALL_JSON)
print(IDENTIFIERS_JSON)
print(PRICES_JSON)


## Initialize the Database and Import the Sample Catalog

This uses the importer module directly rather than shelling out to the CLI.
That makes it easier to see the functions and their return values in one place.

In [ ]:
initialize_database(DB_PATH)

scryfall_stats = import_scryfall_cards(DB_PATH, SCRYFALL_JSON)
identifier_stats = import_mtgjson_identifiers(DB_PATH, IDENTIFIERS_JSON)
price_stats = import_mtgjson_prices(DB_PATH, PRICES_JSON)

print("scryfall:", scryfall_stats)
print("identifiers:", identifier_stats)
print("prices:", price_stats)


## Inspect the Imported Catalog Tables

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    cards = [
        dict(row)
        for row in conn.execute(
            """
            SELECT scryfall_id, mtgjson_uuid, name, set_code, collector_number,
                   tcgplayer_product_id, cardkingdom_id, cardmarket_id
            FROM mtg_cards
            ORDER BY name
            """
        )
    ]

pprint(cards)


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    prices = [
        dict(row)
        for row in conn.execute(
            """
            SELECT scryfall_id, provider, price_kind, finish, currency, snapshot_date, price_value
            FROM price_snapshots
            ORDER BY scryfall_id, provider, price_kind, finish
            """
        )
    ]

pprint(prices)


## Create a Personal Inventory

In [ ]:
create_inventory(
    DB_PATH,
    slug="personal",
    display_name="Personal Collection",
    description="Binders, decks, and loose cards",
)

inventory_rows = list_inventories(DB_PATH)
print(render_table(
    inventory_rows,
    [
        ("slug", "slug"),
        ("display_name", "name"),
        ("description", "description"),
        ("item_rows", "item_rows"),
        ("total_cards", "total_cards"),
    ],
))


## Search the Local Card Catalog

This uses the same local search helper that the thin CLI uses, with table output
so it is easier to compare candidate printings.

In [ ]:
search_columns = [
    ("name", "name"),
    ("set_code", "set"),
    ("set_name", "set_name"),
    ("collector_number", "number"),
    ("lang", "lang"),
    ("rarity", "rarity"),
    ("finishes", "finishes"),
    ("scryfall_id", "scryfall_id"),
]

bolt_matches = search_cards(DB_PATH, query="Lightning Bolt", set_code=None, exact=False, limit=10)
print("Lightning Bolt matches")
print(render_table(bolt_matches, search_columns))
print()

ring_matches = search_cards(DB_PATH, query="Sol Ring", set_code="clb", exact=True, limit=10)
print("Sol Ring exact matches in CLB")
print(render_table(ring_matches, search_columns))


## Add Owned Cards

We add four non-foil `Lightning Bolt` copies and one foil `Sol Ring`.
The notebook now uses the same human-readable confirmation formatter as the CLI.


In [ ]:
bolt_result = add_card(
    DB_PATH,
    inventory_slug="personal",
    scryfall_id="s1",
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=4,
    condition_code="NM",
    finish="normal",
    language_code="en",
    location="Red Binder",
    acquisition_price=2.00,
    acquisition_currency="USD",
    notes="Playset",
)

ring_result = add_card(
    DB_PATH,
    inventory_slug="personal",
    scryfall_id="s2",
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=1,
    condition_code="LP",
    finish="foil",
    language_code="en",
    location="Commander Deck Box",
    acquisition_price=3.50,
    acquisition_currency="USD",
    notes="One shiny copy",
)

print(format_add_card_result(bolt_result))
print()
print(format_add_card_result(ring_result))


## Review the Personal Inventory

In [ ]:
owned_rows = list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)
print(format_owned_rows(owned_rows))


In [ ]:
valuation_rows = valuation(DB_PATH, inventory_slug="personal", provider="tcgplayer")
print(render_table(
    valuation_rows,
    [
        ("provider", "provider"),
        ("currency", "currency"),
        ("item_rows", "item_rows"),
        ("total_cards", "total_cards"),
        ("total_value", "total_value"),
    ],
))


## Run the Thin CLI from the Notebook

The package functions are useful for Python-driven workflows, but the repo also
ships thin terminal commands. This cell shows the same valuation through the CLI
wrapper script, which keeps working even if you have not activated an editable
install in the notebook kernel.

In [ ]:
valuation_cmd = [
    sys.executable,
    str(STACK_DIR / "personal_inventory_cli.py"),
    "valuation",
    "--db",
    str(DB_PATH),
    "--inventory",
    "personal",
    "--provider",
    "tcgplayer",
]

completed = subprocess.run(
    valuation_cmd,
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(completed.stdout)


## What This Notebook Demonstrated

At this point you have seen the whole local-first loop working:

- the packaged importer creates and fills the catalog tables
- vendor identifiers and daily price snapshots land in SQLite
- the personal inventory layer points at exact printings
- valuation comes from the latest imported price snapshots

Good next expansions would be:

- richer search filters for set, rarity, and finish
- movement history and acquisition analytics
- a terminal UI or lightweight web UI on top of the same database